# 26 — Diverse Bagging Ensemble for Deployable Firewall

**Objective**: Implement "Diverse Bagging" strategy for production-ready VPN firewall with Zero-FP guarantee.

## Strategy:
1. **Robust 9 Features**: Direction-invariant packet-size features with Coefficient of Variation
2. **Sub-Capture Splitting**: USBVPN split with protocol stratification (SUBCAPTURE_SIZE=100)
3. **DIVERSE BAGGING**: 3 model families (XGB, LGBM, Cat) × 3 diverse bags = 9 models
   - Bag 1 (Sensitive): 1:1 ratio - High recall for VPN detection
   - Bag 2 (Balanced): 1:5 ratio - Balanced detection
   - Bag 3 (Conservative): 1:10 ratio - Safety brake preventing false positives in enterprise
   - Conservative bag prevents aggressive bags from causing FP in production
4. **Forensic Validation**: Leave-One-Dataset-Out (LOOD) tests to verify cross-era generalization

In [1]:
%matplotlib inline

import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

sys.path.append(str(Path.cwd().parent))

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline
from src.models.train_balanced_bagging_ensemble import run_balanced_bagging

paths = load_paths()
logger = setup_logger(level="INFO")

SEED = 42

# THE ROBUST 9: Direction-invariant packet-size features only
# Using Coefficient of Variation (CV) instead of raw std for unitless burstiness
ROBUST_FEATURES = [
    "sz_all_mean",
    "sz_cv",  # Coefficient of Variation: sz_all_std / sz_all_mean (unitless)
    "sz_all_p25",
    "sz_all_median",
    "sz_all_p75",
    "sz_mean_max",
    "sz_mean_min",
    "sz_std_max",
    "sz_std_min",
]

print(f"Using Robust 9 features: {ROBUST_FEATURES}")

Using Robust 9 features: ['sz_all_mean', 'sz_cv', 'sz_all_p25', 'sz_all_median', 'sz_all_p75', 'sz_mean_max', 'sz_mean_min', 'sz_std_max', 'sz_std_min']


## 1. Load Multi-Domain Data

In [2]:
df_all = load_and_prepare_data()

print(f"Multi-Domain Pool: {df_all.shape}")
print("\nDatasets by split:")
print(pd.crosstab(df_all["split"], df_all["dataset"]))

print("\nLabel distribution:")
print(pd.crosstab(df_all["dataset"], df_all["label"], margins=True))

df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()
df_test = df_all[df_all["split"] == "test"].copy()

# Fit pipeline and verify Robust 9 are present
pipe = FeaturePipeline().fit(df_train)
all_features = pipe.model_feature_names()

missing_robust = [f for f in ROBUST_FEATURES if f not in all_features]
if missing_robust:
    raise ValueError(f"Missing Robust 9 features: {missing_robust}")

print(f"\nFeature pipeline fitted with {len(all_features)} features")
print(f"Robust 9 confirmed present: {all(f in all_features for f in ROBUST_FEATURES)}")

2026-03-25 18:20:09 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-25 18:20:38 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-25 18:21:27 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-25 18:21:27 | INFO | ai-vpn-firewall | Metadata columns present for analysis only: ['source_capture_id', 'source_file']
2026-03-25 18:21:27 | INFO | ai-vpn-firewall | Multi-Domain Pool Created: (69558, 49)
2026-03-25 18:21:27 | INFO | ai-vpn-firewall | Datasets: {'usbvpn': 49650, 'iscx': 11801, 'vnat': 8107}
2026-03-25 18:21:27 | INFO | ai-vpn-firewall | Splits: {'train': 51499, 'test': 9384, 'val': 8675}
Multi-Domain Pool: (69558, 49)

Datasets by split:
dataset  iscx  usbvpn  vnat
split                      
test     1634    7540   210
train    8388   35361  7750
val      1779    6749   147

Label distribution:
label        0      1    All
dataset                     
iscx      8858   2943  11801
usbvpn   42487   7163  49650
vnat      7733    374   8

## 2. Train Balanced Bagging Ensemble

**Configuration**:
- 3 model families: XGBoost, LightGBM, CatBoost
- 3 bags per family (different random seeds)
- Each bag: 100% VPN + 1:1 downsampled Benign
- Total: 9 models averaged

In [3]:
# Transform data with pipeline
df_train_transformed = pipe.transform(df_train)
df_val_transformed = pipe.transform(df_val)
df_test_transformed = pipe.transform(df_test)

# Combine back for the balanced bagging function
df_all_transformed = pd.concat([df_train_transformed, df_val_transformed, df_test_transformed], ignore_index=True)

print(f"Transformed data shape: {df_all_transformed.shape}")
print(f"Training ensemble on ROBUST 9 features only...")

output_dir = paths.artifacts_ensemble / "diverse_bagging_robust9"
output_dir.mkdir(parents=True, exist_ok=True)

# DIVERSE BAGGING: Professor's Strategy for Deployable Firewall
# Bag 1 (Sensitive): 1:1 ratio - High recall, catches most VPN
# Bag 2 (Balanced): 1:5 ratio - Balanced detection
# Bag 3 (Conservative): 1:10 ratio - Safety brake for enterprise FPR
results = run_balanced_bagging(
    df=df_all_transformed,
    label_col="label",
    group_col="capture_id",
    dataset_col="dataset",
    split_col="split",
    diverse_ratios=[1.0, 5.0, 10.0],  # Sensitive, Balanced, Conservative
    target_fprs="0.001,0.005,0.01",
    seed=SEED,
    output_dir=str(output_dir),
    model_types=["xgb", "lgbm", "cat"],
    feature_cols=ROBUST_FEATURES,
    weight_xgb=1.0,
    weight_lgbm=1.0,
    weight_cat=1.0,
)

print("\n" + "="*80)
print("BALANCED ENSEMBLE TRAINING COMPLETE")
print("="*80)

Transformed data shape: (69558, 45)
Training ensemble on ROBUST 9 features only...

BALANCED ENSEMBLE TRAINING COMPLETE


## 3. Forensic Validation: Leave-One-Dataset-Out (LOOD)

This is the critical test to verify we've solved the era gap problem.

In [4]:
def run_lood_test(df_all_in, pipe, feature_cols, seed=42):
    """
    Leave-One-Dataset-Out cross-validation using Balanced Ensemble.
    """
    from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
    
    datasets = sorted(df_all_in["dataset"].unique())
    results = []
    
    for held_out in datasets:
        print(f"\n{'='*60}")
        print(f"LOOD: Holding out {held_out.upper()}")
        print(f"{'='*60}")
        
        # Create splits
        train_mask = (df_all_in["split"] == "train") & (df_all_in["dataset"] != held_out)
        val_mask = (df_all_in["split"] == "val") & (df_all_in["dataset"] != held_out)
        test_mask = (df_all_in["split"] == "test") & (df_all_in["dataset"] == held_out)
        
        df_train_lo = df_all_in[train_mask].copy()
        df_val_lo = df_all_in[val_mask].copy()
        df_test_lo = df_all_in[test_mask].copy()
        
        if len(df_train_lo) == 0 or len(df_test_lo) == 0:
            print(f"Skipping {held_out}: insufficient data")
            continue
        
        # Refit pipeline on this LOOD train
        pipe_lo = FeaturePipeline().fit(df_train_lo)
        
        df_train_lo_t = pipe_lo.transform(df_train_lo)
        df_val_lo_t = pipe_lo.transform(df_val_lo)
        df_test_lo_t = pipe_lo.transform(df_test_lo)
        
        # Combine for balanced bagging
        df_lo_combined = pd.concat([df_train_lo_t, df_val_lo_t, df_test_lo_t], ignore_index=True)
        
        # Train balanced ensemble
        temp_dir = paths.artifacts_ensemble / f"lood_{held_out}"
        temp_dir.mkdir(parents=True, exist_ok=True)
        
        lood_results = run_balanced_bagging(
            df=df_lo_combined,
            label_col="label",
            group_col="capture_id",
            dataset_col="dataset",
            split_col="split",
            bags_per_family=3,
            majority_ratio=1.0,
            target_fprs="0.01",
            seed=seed,
            output_dir=str(temp_dir),
            model_types=["xgb", "lgbm", "cat"],
            feature_cols=feature_cols,
            weight_xgb=1.0,
            weight_lgbm=1.0,
            weight_cat=1.0,
        )
        
        # Extract test metrics
        test_metrics = lood_results["isotonic"]["test_overall"]
        
        results.append({
            "train_on": "+".join(sorted([d for d in datasets if d != held_out])),
            "test_on": held_out,
            "auc": test_metrics["auc"],
            "pr_auc": test_metrics["pr_auc"],
            "threshold": test_metrics["fpr_0.01"]["threshold"],
            "recall": test_metrics["fpr_0.01"]["recall"],
            "precision": test_metrics["fpr_0.01"]["precision"],
            "fpr": test_metrics["fpr_0.01"]["fpr"],
        })
    
    return pd.DataFrame(results)


print("\n" + "="*80)
print("FORENSIC VALIDATION: LEAVE-ONE-DATASET-OUT")
print("="*80)

lood_df = run_lood_test(df_all, pipe, ROBUST_FEATURES, seed=SEED)

print("\n" + "="*80)
print("LOOD RESULTS")
print("="*80)
print(lood_df.to_string(index=False, float_format="%.4f"))

mean_auc = lood_df["auc"].mean()
print(f"\n Mean LOOD AUC: {mean_auc:.4f}")

if mean_auc > 0.80:
    print(" SUCCESS: Era-independent model achieved robust generalization (>0.80)")
else:
    print(f"  WARNING: LOOD AUC still below 0.80 threshold")


FORENSIC VALIDATION: LEAVE-ONE-DATASET-OUT

LOOD: Holding out ISCX

LOOD: Holding out USBVPN

LOOD: Holding out VNAT

LOOD RESULTS
   train_on test_on    auc  pr_auc  threshold  recall  precision    fpr
usbvpn+vnat    iscx 0.3390  0.1307     0.1333  0.0630     0.0225 0.4663
  iscx+vnat  usbvpn 0.3074  0.1075     0.7826  0.0241     0.0335 0.1110
iscx+usbvpn    vnat 0.5877  0.2004     0.2963  0.4615     0.0789 0.3553

 Mean LOOD AUC: 0.4113


## 4. Detailed Test Results by Dataset

In [5]:
print("\n" + "="*80)
print("DETAILED TEST PERFORMANCE BY DATASET (Isotonic Calibration)")
print("="*80)

# Load predictions from the main ensemble
pred_df = pd.read_csv(output_dir / "predictions.csv")
test_pred = pred_df[pred_df["split"] == "test"].copy()

summary_rows = []

for ds in sorted(test_pred["dataset"].unique()):
    sub = test_pred[test_pred["dataset"] == ds]
    y_true = sub["label"].values
    y_prob = sub["prob_iso"].values
    
    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    
    # Use FPR=0.01 threshold from results
    with open(output_dir / "metrics.json", "r") as f:
        metrics = json.load(f)
    
    thr = metrics["isotonic"][f"test_{ds}"]["fpr_0.01"]["threshold"]
    y_pred = (y_prob >= thr).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    summary_rows.append({
        "dataset": ds,
        "auc": auc,
        "pr_auc": pr_auc,
        "threshold": thr,
        "recall": recall,
        "precision": precision,
        "fpr": fpr,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "n_test": len(y_true),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format="%.4f"))


DETAILED TEST PERFORMANCE BY DATASET (Isotonic Calibration)
dataset    auc  pr_auc  threshold  recall  precision    fpr   tn  fp  fn   tp  n_test
   iscx 0.9175  0.7946     0.5714  0.5504     0.9161 0.0086 1384  12 107  131    1634
 usbvpn 0.9991  0.9940     0.5714  0.9923     0.9313 0.0117 6426  76   8 1030    7540
   vnat 0.9990  0.9779     0.5714  1.0000     0.8667 0.0102  195   2   0   13     210


## 5. Final Summary Report

In [6]:
print("\n" + "="*80)
print("ROBUST BALANCED ENSEMBLE: FINAL SUMMARY")
print("="*80)

print("\n1. STRATEGY IMPLEMENTATION:")
print(f"   ✓ Robust 9 Features: {ROBUST_FEATURES}")
print(f"   ✓ USBVPN Sub-capture size: 100 (increased test VPN representation)")
print(f"   ✓ DIVERSE Bagging: 3 families × 3 diverse bags = 9 models")
print(f"   ✓ Bag 1 (Sensitive): 1:1 ratio - High recall")
print(f"   ✓ Bag 2 (Balanced): 1:5 ratio - Balanced detection")
print(f"   ✓ Bag 3 (Conservative): 1:10 ratio - Safety brake for FPR")

print("\n2. FORENSIC VALIDATION (LOOD):")
print(lood_df[["train_on", "test_on", "auc"]].to_string(index=False))
print(f"\n   Mean LOOD AUC: {lood_df['auc'].mean():.4f}")
print(f"   Min LOOD AUC:  {lood_df['auc'].min():.4f}")
print(f"   Max LOOD AUC:  {lood_df['auc'].max():.4f}")

print("\n3. PER-DATASET TEST PERFORMANCE:")
print(summary_df[["dataset", "auc", "pr_auc", "recall", "precision", "fpr"]].to_string(index=False))

print("\n4. LEAKAGE VERIFICATION:")
# Check capture_id overlap
train_caps = set(df_train["capture_id"].unique())
test_caps = set(df_test["capture_id"].unique())
overlap = train_caps & test_caps
print(f"   Train/Test capture_id overlap: {len(overlap)} (should be 0)")

if len(overlap) == 0:
    print("  No leakage detected")
else:
    print(f"  WARNING: {len(overlap)} capture_ids appear in both train and test")

print("\n5. DELIVERABLES:")
print(f"   - Models saved to: {output_dir}")
print(f"   - Predictions: {output_dir / 'predictions.csv'}")
print(f"   - Metrics: {output_dir / 'metrics.json'}")

print("\n" + "="*80)
if lood_df["auc"].mean() > 0.80:
    print(" MISSION ACCOMPLISHED: Era-independent VPN detection achieved!")
else:
    print("Results logged. Further optimization may be needed.")
print("="*80)


ROBUST BALANCED ENSEMBLE: FINAL SUMMARY

1. STRATEGY IMPLEMENTATION:
   ✓ Robust 9 Features: ['sz_all_mean', 'sz_cv', 'sz_all_p25', 'sz_all_median', 'sz_all_p75', 'sz_mean_max', 'sz_mean_min', 'sz_std_max', 'sz_std_min']
   ✓ USBVPN Sub-capture size: 100 (increased test VPN representation)
   ✓ DIVERSE Bagging: 3 families × 3 diverse bags = 9 models
   ✓ Bag 1 (Sensitive): 1:1 ratio - High recall
   ✓ Bag 2 (Balanced): 1:5 ratio - Balanced detection
   ✓ Bag 3 (Conservative): 1:10 ratio - Safety brake for FPR

2. FORENSIC VALIDATION (LOOD):
   train_on test_on      auc
usbvpn+vnat    iscx 0.338986
  iscx+vnat  usbvpn 0.307382
iscx+usbvpn    vnat 0.587661

   Mean LOOD AUC: 0.4113
   Min LOOD AUC:  0.3074
   Max LOOD AUC:  0.5877

3. PER-DATASET TEST PERFORMANCE:
dataset      auc   pr_auc   recall  precision      fpr
   iscx 0.917462 0.794579 0.550420   0.916084 0.008596
 usbvpn 0.999099 0.993998 0.992293   0.931284 0.011689
   vnat 0.999024 0.977909 1.000000   0.866667 0.010152

4. LE